In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# GRU Baseline for Environmental Noise Cancellation

import os
import glob
import time
import random

import numpy as np
import tensorflow as tf

# DETERMINISTIC EXPERIMENT CONTROLS
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# EXPERIMENT CONFIGURATION
EXPERIMENT_NAME = "GRU_baseline"

DATASET_PATH = "/content/drive/MyDrive/T-MLP/processed_data"

TRAIN_PATH = os.path.join(
    DATASET_PATH,
    "train"
)

VAL_PATH = os.path.join(
    DATASET_PATH,
    "validation"
)

TEST_PATH = os.path.join(
    DATASET_PATH,
    "test"
)


# Expected dataset size
EXPECTED_TRAIN = 800
EXPECTED_VAL = 100
EXPECTED_TEST = 100


# Audio window configuration
WINDOW_SIZE = 256
HOP_SIZE = 128

# TRAINING CONFIGURATION
BATCH_SIZE = 32

# Keep identical to MLP and T-MLP experiments
EPOCHS = 10

# Lower learning rate than the original 0.01
# to prevent GRU training instability.
LEARNING_RATE = 0.001

# Prevent exploding gradients
CLIPNORM = 1.0

# MODEL SAVE PATH
SAVE_DIR = "/content/drive/MyDrive/T-MLP/saved_models"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

MODEL_SAVE_PATH = os.path.join(
    SAVE_DIR,
    "GRU_best.keras"
)

# ENVIRONMENT
print("\n========================================")
print("GRU ENVIRONMENT")
print("========================================")

print(
    "Experiment:",
    EXPERIMENT_NAME
)

print(
    "TensorFlow:",
    tf.__version__
)

print(
    "Random seed:",
    SEED
)

# GPU CHECK
print("\nAvailable GPUs:")

gpus = tf.config.list_physical_devices("GPU")

if gpus:

    for gpu in gpus:
        print(" ", gpu)

else:

    print("WARNING: No GPU detected.")


print(
    "\nDeterministic operations: ENABLED"
)

# DATASET VERIFICATION
print("\n========================================")
print("DATASET VERIFICATION")
print("========================================")


train_files = sorted(
    glob.glob(
        os.path.join(
            TRAIN_PATH,
            "*.npz"
        )
    )
)

val_files = sorted(
    glob.glob(
        os.path.join(
            VAL_PATH,
            "*.npz"
        )
    )
)

test_files = sorted(
    glob.glob(
        os.path.join(
            TEST_PATH,
            "*.npz"
        )
    )
)


print(
    f"Training samples   : {len(train_files)}"
)

print(
    f"Validation samples : {len(val_files)}"
)

print(
    f"Test samples       : {len(test_files)}"
)


if len(train_files) != EXPECTED_TRAIN:

    raise ValueError(
        f"Expected {EXPECTED_TRAIN} training samples, "
        f"found {len(train_files)}"
    )


if len(val_files) != EXPECTED_VAL:

    raise ValueError(
        f"Expected {EXPECTED_VAL} validation samples, "
        f"found {len(val_files)}"
    )


if len(test_files) != EXPECTED_TEST:

    raise ValueError(
        f"Expected {EXPECTED_TEST} test samples, "
        f"found {len(test_files)}"
    )


print(
    "Dataset verification successful."
)

# LOAD SLIDING WINDOWS
def load_windows(file_list):

    X = []
    Y = []

    for file_path in file_list:

        data = np.load(
            file_path
        )

        noisy = data[
            "noisy_speech"
        ].astype(
            np.float32
        )

        clean = data[
            "clean_speech"
        ].astype(
            np.float32
        )

        # Length check
        if len(noisy) != len(clean):

            raise ValueError(
                f"Length mismatch in {file_path}"
            )

        # NaN / Inf check
        if not np.all(
            np.isfinite(noisy)
        ):

            raise ValueError(
                f"NaN/Inf detected in noisy speech: "
                f"{file_path}"
            )


        if not np.all(
            np.isfinite(clean)
        ):

            raise ValueError(
                f"NaN/Inf detected in clean speech: "
                f"{file_path}"
            )

        # Sliding windows
        for start in range(
            0,
            len(noisy) - WINDOW_SIZE + 1,
            HOP_SIZE
        ):

            end = start + WINDOW_SIZE

            X.append(
                noisy[start:end]
            )

            Y.append(
                clean[start:end]
            )


    X = np.asarray(
        X,
        dtype=np.float32
    )

    Y = np.asarray(
        Y,
        dtype=np.float32
    )


    return X, Y

# LOAD TRAINING DATA
print("\n========================================")
print("LOADING TRAINING DATA")
print("========================================")


X_train, Y_train = load_windows(
    train_files
)


print(
    f"Training windows: {len(X_train)}"
)

print(
    f"X_train shape   : {X_train.shape}"
)

print(
    f"Y_train shape   : {Y_train.shape}"
)

# LOAD VALIDATION DATA
print("\n========================================")
print("LOADING VALIDATION DATA")
print("========================================")


X_val, Y_val = load_windows(
    val_files
)


print(
    f"Validation windows: {len(X_val)}"
)

print(
    f"X_val shape       : {X_val.shape}"
)

print(
    f"Y_val shape       : {Y_val.shape}"
)

# LOAD TEST DATA
print("\n========================================")
print("LOADING TEST DATA")
print("========================================")


X_test, Y_test = load_windows(
    test_files
)


print(
    f"Test windows: {len(X_test)}"
)

print(
    f"X_test shape: {X_test.shape}"
)

print(
    f"Y_test shape: {Y_test.shape}"
)



# PREPARE DATA FOR GRU

# Current audio window:
#
#     (batch, 256)
#
# GRU expects:
#
#     (batch, timesteps, features)
#
# Therefore:
#
#     (batch, 256, 1)


X_train = X_train[
    ...,
    np.newaxis
]

X_val = X_val[
    ...,
    np.newaxis
]

X_test = X_test[
    ...,
    np.newaxis
]


Y_train = Y_train[
    ...,
    np.newaxis
]

Y_val = Y_val[
    ...,
    np.newaxis
]

Y_test = Y_test[
    ...,
    np.newaxis
]


print("\n========================================")
print("GRU INPUT SHAPES")
print("========================================")


print(
    "X_train:",
    X_train.shape
)

print(
    "Y_train:",
    Y_train.shape
)

print(
    "X_val  :",
    X_val.shape
)

print(
    "Y_val  :",
    Y_val.shape
)

print(
    "X_test :",
    X_test.shape
)

print(
    "Y_test :",
    Y_test.shape
)

# TF.DATA PIPELINE
print("\n========================================")
print("CREATING TF.DATA PIPELINE")
print("========================================")


train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        X_train,
        Y_train
    )
)


train_dataset = (
    train_dataset
    .shuffle(
        buffer_size=len(X_train),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)


val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        X_val,
        Y_val
    )
)


val_dataset = (
    val_dataset
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)

# BUILD GRU MODEL
def build_gru():

    inputs = tf.keras.Input(
        shape=(
            WINDOW_SIZE,
            1
        ),
        name="noisy_audio"
    )

    # Layer 1
    x = tf.keras.layers.GRU(
        64,
        return_sequences=True,
        name="gru_1"
    )(inputs)

    # GRU Layer 2
    x = tf.keras.layers.GRU(
        64,
        return_sequences=True,
        name="gru_2"
    )(x)

    # Audio reconstruction
    outputs = tf.keras.layers.Dense(
        1,
        activation=None,
        name="audio_output"
    )(x)


    return tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="GRU"
    )

# BUILD MODEL
print("\n========================================")
print("BUILDING GRU MODEL")
print("========================================")


model = build_gru()


model.summary()


print(
    f"\nTotal parameters: "
    f"{model.count_params():,}"
)

# COMPILE
print("\n========================================")
print("COMPILING MODEL")
print("========================================")


optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE,
    clipnorm=CLIPNORM
)


model.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)


print(
    "Optimizer     : Adam"
)

print(
    "Learning rate :",
    LEARNING_RATE
)

print(
    "Gradient clip :",
    CLIPNORM
)

print(
    "Loss          : MSE"
)

# MODEL CHECKPOINT
print("\n========================================")
print("CONFIGURING CHECKPOINT")
print("========================================")


checkpoint = tf.keras.callbacks.ModelCheckpoint(

    filepath=MODEL_SAVE_PATH,

    monitor="val_loss",

    mode="min",

    save_best_only=True,

    save_weights_only=False,

    verbose=1
)


callbacks = [
    checkpoint
]

# TRAIN
print("\n========================================")
print("STARTING GRU TRAINING")
print("========================================")


print(
    f"Epochs        : {EPOCHS}"
)

print(
    f"Batch size    : {BATCH_SIZE}"
)

print(
    f"Learning rate : {LEARNING_RATE}"
)

print(
    f"Model path    : {MODEL_SAVE_PATH}"
)


start_training = time.perf_counter()


history = model.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=EPOCHS,

    callbacks=callbacks,

    verbose=1
)


training_time = (
    time.perf_counter()
    - start_training
)


print("\n========================================")
print("TRAINING COMPLETED")
print("========================================")


print(
    f"Training time: "
    f"{training_time:.2f} seconds"
)


print(
    f"Epochs trained: "
    f"{len(history.history['loss'])}"
)

# LOAD BEST MODEL
print("\n========================================")
print("LOADING BEST GRU MODEL")
print("========================================")


if not os.path.exists(
    MODEL_SAVE_PATH
):

    raise FileNotFoundError(
        "GRU_best.keras was not created."
    )


model = tf.keras.models.load_model(
    MODEL_SAVE_PATH
)


print(
    "Best GRU model loaded successfully."
)

print(
    "Path:",
    MODEL_SAVE_PATH
)

# TEST EVALUATION
print("\n========================================")
print("TEST EVALUATION")
print("========================================")


test_loss, test_mae = model.evaluate(

    X_test,

    Y_test,

    batch_size=BATCH_SIZE,

    verbose=1
)


test_rmse = np.sqrt(
    test_loss
)


print(
    f"\nTest MSE  : {test_loss:.8f}"
)

print(
    f"Test RMSE : {test_rmse:.8f}"
)

print(
    f"Test MAE  : {test_mae:.8f}"
)

# GENERATE TEST PREDICTIONS
print("\n========================================")
print("GENERATING TEST PREDICTIONS")
print("========================================")


start_prediction = time.perf_counter()


Y_pred = model.predict(

    X_test,

    batch_size=BATCH_SIZE,

    verbose=1
)


prediction_time = (
    time.perf_counter()
    - start_prediction
)


print(
    f"Prediction time: "
    f"{prediction_time:.4f} seconds"
)

print(
    "Prediction shape:",
    Y_pred.shape
)

# PREDICTION VALIDATION
print("\n========================================")
print("PREDICTION VALIDATION")
print("========================================")


if not np.all(
    np.isfinite(Y_pred)
):

    raise ValueError(
        "GRU produced NaN/Inf predictions."
    )


print(
    "NaN/Inf check: PASSED"
)

print(
    f"Prediction min : {np.min(Y_pred):.6f}"
)

print(
    f"Prediction max : {np.max(Y_pred):.6f}"
)

print(
    f"Prediction mean: {np.mean(Y_pred):.6f}"
)

print(
    f"Prediction std : {np.std(Y_pred):.6f}"
)

# INFERENCE LATENCY
print("\n========================================")
print("MEASURING INFERENCE LATENCY")
print("========================================")


# Warm-up
_ = model.predict(
    X_test[:1],
    verbose=0
)


latencies = []


for i in range(
    len(X_test)
):

    sample = X_test[
        i:i + 1
    ]


    start = time.perf_counter()


    _ = model.predict(
        sample,
        verbose=0
    )


    end = time.perf_counter()


    latency_ms = (
        end - start
    ) * 1000.0


    latencies.append(
        latency_ms
    )


latencies = np.asarray(
    latencies,
    dtype=np.float64
)


mean_latency = np.mean(
    latencies
)

median_latency = np.median(
    latencies
)

min_latency = np.min(
    latencies
)

max_latency = np.max(
    latencies
)


print(
    f"Mean latency   : "
    f"{mean_latency:.4f} ms"
)

print(
    f"Median latency : "
    f"{median_latency:.4f} ms"
)

print(
    f"Min latency    : "
    f"{min_latency:.4f} ms"
)

print(
    f"Max latency    : "
    f"{max_latency:.4f} ms"
)

# MODEL SIZE
model_size_mb = (
    os.path.getsize(
        MODEL_SAVE_PATH
    )
    / (1024 ** 2)
)

# FINAL RESULTS
print("\n========================================")
print("GRU FINAL RESULTS")
print("========================================")


print(
    f"Experiment name    : "
    f"{EXPERIMENT_NAME}"
)

print(
    f"Random seed        : "
    f"{SEED}"
)

print(
    f"Training samples   : "
    f"{EXPECTED_TRAIN}"
)

print(
    f"Validation samples : "
    f"{EXPECTED_VAL}"
)

print(
    f"Test samples       : "
    f"{EXPECTED_TEST}"
)

print(
    f"Training windows   : "
    f"{len(X_train)}"
)

print(
    f"Validation windows : "
    f"{len(X_val)}"
)

print(
    f"Test windows       : "
    f"{len(X_test)}"
)

print(
    f"Window size        : "
    f"{WINDOW_SIZE}"
)

print(
    f"Hop size           : "
    f"{HOP_SIZE}"
)

print(
    f"Epochs             : "
    f"{EPOCHS}"
)

print(
    f"Batch size         : "
    f"{BATCH_SIZE}"
)

print(
    f"Learning rate      : "
    f"{LEARNING_RATE}"
)

print(
    f"Gradient clipnorm  : "
    f"{CLIPNORM}"
)

print(
    f"Parameters         : "
    f"{model.count_params():,}"
)

print(
    f"Test MSE           : "
    f"{test_loss:.8f}"
)

print(
    f"Test RMSE          : "
    f"{test_rmse:.8f}"
)

print(
    f"Test MAE           : "
    f"{test_mae:.8f}"
)

print(
    f"Mean latency       : "
    f"{mean_latency:.4f} ms"
)

print(
    f"Median latency     : "
    f"{median_latency:.4f} ms"
)

print(
    f"Training time      : "
    f"{training_time:.2f} sec"
)

print(
    f"Model size         : "
    f"{model_size_mb:.4f} MB"
)


print("\n----------------------------------------")
print("MODEL SAVED TO:")
print(MODEL_SAVE_PATH)
print("----------------------------------------")


print("\n========================================")
print("GRU TRAINING FINISHED")
print("========================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

GRU ENVIRONMENT
Experiment: GRU_baseline
TensorFlow: 2.21.0
Random seed: 42

Available GPUs:
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

Deterministic operations: ENABLED

DATASET VERIFICATION
Training samples   : 800
Validation samples : 100
Test samples       : 100
Dataset verification successful.

LOADING TRAINING DATA
Training windows: 732388
X_train shape   : (732388, 256)
Y_train shape   : (732388, 256)

LOADING VALIDATION DATA
Validation windows: 80583
X_val shape       : (80583, 256)
Y_val shape       : (80583, 256)

LOADING TEST DATA
Test windows: 85649
X_test shape: (85649, 256)
Y_test shape: (85649, 256)

GRU INPUT SHAPES
X_train: (732388, 256, 1)
Y_train: (732388, 256, 1)
X_val  : (80583, 256, 1)
Y_val  : (80583, 256, 1)
X_test : (85649, 256, 1)
Y_test : (85649, 256, 1)

CREATING TF.DATA PIPELINE

BUILDING GRU MODEL


Model: "GRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ noisy_audio (InputLayer)        │ (None, 256, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 256, 64)        │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 256, 64)        │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ audio_output (Dense)            │ (None, 256, 1)         │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,889 (148.00 KB)

 Trainable params: 37,889 (148.00 KB)

 Non-trainable params: 0 (0.00 B)


Total parameters: 37,889

COMPILING MODEL
Optimizer     : Adam
Learning rate : 0.001
Gradient clip : 1.0
Loss          : MSE

CONFIGURING CHECKPOINT

STARTING GRU TRAINING
Epochs        : 10
Batch size    : 32
Learning rate : 0.001
Model path    : /content/drive/MyDrive/T-MLP/saved_models/GRU_best.keras
Epoch 1/10
22888/22888 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0016 - mae: 0.0226
Epoch 1: val_loss improved from None to 0.00089, saving model to /content/drive/MyDrive/T-MLP/saved_models/GRU_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/T-MLP/saved_models/GRU_best.keras
22888/22888 ━━━━━━━━━━━━━━━━━━━━ 479s 21ms/step - loss: 0.0014 - mae: 0.0209 - val_loss: 8.9136e-04 - val_mae: 0.0167
Epoch 2/10
22887/22888 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0011 - mae: 0.0184
Epoch 2: val_loss improved from 0.00089 to 0.00081, saving model to /content/drive/MyDrive/T-MLP/saved_models/GRU_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/T-MLP/s